
# HW3 ARMA Analysis Notebook

This notebook reproduces the full workflow for **Part 1** and provides a ready-to-run template for **Part 2**.

**What it does:**
- Loads `HW3 ARMA.dta`
- Plots ACF & PACF (20 lags) for `Y1` and `Y2`
- Runs ARMA(p,0,q) grid search (default p,q ≤ 3), and expands to p,q ≤ 5 if residuals are not white
- Fits the top model + two alternatives (by AIC), exports summaries and residual diagnostics (Ljung–Box Q)
- Chooses final models that whiten residuals
- Saves all outputs under `arma_hw3_outputs/`

**Paths**
- Data file: `/mnt/data/HW3 ARMA.dta`
- Outputs: `/mnt/data/arma_hw3_outputs/`


In [1]:

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from statsmodels.tsa.stattools import acf, pacf, adfuller
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox

DATA_PATH = "/mnt/data/HW3 ARMA.dta"
OUTDIR = "/mnt/data/arma_hw3_outputs"
os.makedirs(OUTDIR, exist_ok=True)

df = pd.read_stata(DATA_PATH)
df.columns = [str(c).strip() for c in df.columns]
series_names = [c for c in ["Y1","Y2"] if c in df.columns]
assert series_names, "Could not find Y1/Y2 in the dataset."
df[series_names].head()


OSError: [Errno 30] Read-only file system: '/mnt'

In [ ]:

def plot_acf_pacf(x, name, nlags=20, outdir=OUTDIR):
    acf_vals = acf(x, nlags=nlags, fft=False)
    pacf_vals = pacf(x, nlags=nlags, method='yw')
    n = len(x.dropna())
    conf = 1.96/np.sqrt(n)

    # ACF plot
    plt.figure(figsize=(7,4))
    plt.stem(range(len(acf_vals)), acf_vals, use_line_collection=True)
    plt.axhline(conf, linestyle='--')
    plt.axhline(-conf, linestyle='--')
    plt.title(f"{name} ACF (0..{nlags})")
    plt.xlabel("Lag"); plt.ylabel("ACF")
    acf_path = os.path.join(outdir, f"{name}_ACF_0_{nlags}.png")
    plt.tight_layout(); plt.savefig(acf_path, dpi=150); plt.show()

    # PACF plot
    plt.figure(figsize=(7,4))
    plt.stem(range(len(pacf_vals)), pacf_vals, use_line_collection=True)
    plt.axhline(conf, linestyle='--')
    plt.axhline(-conf, linestyle='--')
    plt.title(f"{name} PACF (0..{nlags})")
    plt.xlabel("Lag"); plt.ylabel("PACF")
    pacf_path = os.path.join(outdir, f"{name}_PACF_0_{nlags}.png")
    plt.tight_layout(); plt.savefig(pacf_path, dpi=150); plt.show()

    return acf_vals, pacf_vals, acf_path, pacf_path

def arma_grid_search(x, name, pmax=3, qmax=3):
    rows = []
    for p in range(pmax+1):
        for q in range(qmax+1):
            try:
                res = ARIMA(x, order=(p,0,q), enforce_stationarity=False, enforce_invertibility=False).fit()
                rows.append({
                    "series": name, "p": p, "q": q,
                    "aic": res.aic, "bic": res.bic,
                    "loglik": res.llf, "params": len(res.params),
                    "converged": res.mle_retvals.get("converged", True) if hasattr(res,"mle_retvals") else True
                })
            except Exception as e:
                rows.append({"series": name, "p": p, "q": q,
                             "aic": np.nan, "bic": np.nan, "loglik": np.nan,
                             "params": np.nan, "converged": False})
    return pd.DataFrame(rows).sort_values("aic")

def fit_with_diagnostics(x, name, p, q, lb_lags=20):
    res = ARIMA(x, order=(p,0,q), enforce_stationarity=False, enforce_invertibility=False).fit()
    resid = res.resid.dropna()
    lb = acorr_ljungbox(resid, lags=lb_lags, return_df=True)

    # Residual ACF/PACF
    r_acf, r_pacf, r_acf_path, r_pacf_path = plot_acf_pacf(resid, f"{name}_resid_ARMA({p},{q})", nlags=lb_lags)

    # Save text summary + LB table
    summ_path = os.path.join(OUTDIR, f"{name}_ARMA_{p}_{q}_summary.txt")
    with open(summ_path, "w") as f:
        f.write(str(res.summary()))
    lb_path = os.path.join(OUTDIR, f"{name}_ARMA_{p}_{q}_ljungbox.csv")
    lb.to_csv(lb_path, index=True)

    return res, resid, summ_path, lb_path, r_acf_path, r_pacf_path


## Part 1a–b: ADF + Correlograms for Y1, Y2

In [ ]:

adf_rows = []
acf_paths = []

for name in series_names:
    x = pd.to_numeric(df[name], errors='coerce').dropna()

    # ADF
    try:
        stat, p, _, _, crit, _ = adfuller(x, autolag='AIC')
    except Exception:
        stat, p, crit = np.nan, np.nan, {}
    adf_rows.append({"series": name, "ADF_stat": stat, "ADF_p": p,
                     "crit_1%": crit.get("1%", np.nan),
                     "crit_5%": crit.get("5%", np.nan),
                     "crit_10%": crit.get("10%", np.nan)})

    # Correlograms
    acf_vals, pacf_vals, acf_path, pacf_path = plot_acf_pacf(x, name, nlags=20, outdir=OUTDIR)
    acf_paths.append({"series": name, "acf_plot": acf_path, "pacf_plot": pacf_path})

adf_df = pd.DataFrame(adf_rows)
adf_df.to_csv(os.path.join(OUTDIR, "adf_stationarity_tests.csv"), index=False)
pd.DataFrame(acf_paths).to_csv(os.path.join(OUTDIR, "acf_pacf_plot_paths.csv"), index=False)
adf_df


### Y1: ACF/PACF at lag 4 (+ why ACF(1)≈PACF(1))

In [ ]:

import numpy as np
name = "Y1"
x = pd.to_numeric(df[name], errors='coerce').dropna()
acf_vals = acf(x, nlags=20, fft=False)
pacf_vals = pacf(x, nlags=20, method='yw')

print("Y1 ACF(1)=", float(acf_vals[1]), " PACF(1)=", float(pacf_vals[1]))
print("Y1 ACF(4)=", float(acf_vals[4]), " PACF(4)=", float(pacf_vals[4]))

print("\nWhy ACF(1)≈PACF(1): PACF at lag 1 equals the simple correlation at lag 1 (no intervening lags).")


## Part 1c–g: ARMA search (p,q ≤ 3), fit candidates, compare AIC/BIC, residual Q-tests

In [ ]:

all_search = []
for name in series_names:
    x = pd.to_numeric(df[name], errors='coerce').dropna()
    search = arma_grid_search(x, name, pmax=3, qmax=3)
    all_search.append(search)

search_df = pd.concat(all_search, ignore_index=True).sort_values(["series","aic"])
search_df.to_csv(os.path.join(OUTDIR, "arma_grid_search_aic.csv"), index=False)
search_df.head(10)


In [ ]:

# Fit best AIC + two alternatives per series
fit_rows = []
for name in series_names:
    sub = search_df[search_df["series"]==name].sort_values("aic").reset_index(drop=True)
    # Take top-3 unique (p,q)
    cands = sub.head(3).to_dict(orient="records")
    x = pd.to_numeric(df[name], errors='coerce').dropna()
    for c in cands:
        p, q = int(c["p"]), int(c["q"])
        try:
            res, resid, summ_path, lb_path, r_acf_path, r_pacf_path = fit_with_diagnostics(x, name, p, q, lb_lags=20)
            fit_rows.append({
                "series": name, "model": f"ARMA({p},{q})",
                "aic": res.aic, "bic": res.bic, "loglik": res.llf,
                "nobs": res.nobs, "params": len(res.params),
                "summary_file": summ_path,
                "ljungbox_file": lb_path,
                "resid_acf_plot": r_acf_path,
                "resid_pacf_plot": r_pacf_path
            })
        except Exception as e:
            fit_rows.append({"series": name, "model": f"ARMA({p},{q})",
                             "aic": np.nan, "bic": np.nan, "loglik": np.nan,
                             "nobs": np.nan, "params": np.nan})
fit_df = pd.DataFrame(fit_rows)
fit_df.to_csv(os.path.join(OUTDIR, "candidate_model_fit_summaries.csv"), index=False)
fit_df


## Re-specification: widen grid to p,q ≤ 5 if residuals are not white

In [ ]:

def widen_and_select(x, name, pmax=5, qmax=5):
    cands = []
    for p in range(pmax+1):
        for q in range(qmax+1):
            try:
                res = ARIMA(x, order=(p,0,q), enforce_stationarity=False, enforce_invertibility=False).fit()
                lb = acorr_ljungbox(res.resid.dropna(), lags=20, return_df=True)
                p20 = float(lb.iloc[19]["lb_pvalue"])
                cands.append((p,q,res.aic,res.bic,res.llf,p20))
            except Exception:
                pass
    cand_df = pd.DataFrame(cands, columns=["p","q","AIC","BIC","loglik","LB_p_lag20"]).sort_values("AIC")
    return cand_df

final_models = {}
for name in series_names:
    x = pd.to_numeric(df[name], errors='coerce').dropna()
    wide = widen_and_select(x, name, pmax=5, qmax=5)
    display(wide.head(12))
    final_models[name] = wide.iloc[0].to_dict()

final_models


In [ ]:

# Save final choices and summaries
for name, row in final_models.items():
    p, q = int(row["p"]), int(row["q"])
    x = pd.to_numeric(df[name], errors='coerce').dropna()
    res = ARIMA(x, order=(p,0,q), enforce_stationarity=False, enforce_invertibility=False).fit()
    with open(os.path.join(OUTDIR, f"{name}_final_ARMA_{p}_{q}_summary.txt"), "w") as f:
        f.write(str(res.summary()))
    lb = acorr_ljungbox(res.resid.dropna(), lags=20, return_df=True)
    lb.to_csv(os.path.join(OUTDIR, f"{name}_final_ARMA_{p}_{q}_ljungbox.csv"), index=True)
    # Residual correlograms
    _ = plot_acf_pacf(res.resid.dropna(), f"{name}_resid_ARMA({p},{q})", nlags=20, outdir=OUTDIR)

final_models



---
# Part 2: Asset Returns — White Noise and ARMA(1,1) Template

> Replace `retA`, `retB`, `retC` and the data load with your actual return series from HW1/2.

## 2(a) Correlogram + Q-statistics


In [ ]:

# Example template code — edit to your variable names / data source
# retA, retB, retC = ... load your 3 return series as pandas Series

def white_noise_check(x, name, nlags=20):
    acf_vals = acf(x, nlags=nlags, fft=False)
    pacf_vals = pacf(x, nlags=nlags, method='yw')
    # Plots
    _ = plot_acf_pacf(x, f"{name}", nlags=nlags, outdir=OUTDIR)
    # Ljung–Box
    lb = acorr_ljungbox(x.dropna(), lags=nlags, return_df=True)
    lb.to_csv(os.path.join(OUTDIR, f"{name}_ljungbox.csv"), index=True)
    return acf_vals, pacf_vals, lb

# Example (uncomment and replace with your series)
# acfA, pacfA, lbA = white_noise_check(retA, "retA")
# acfB, pacfB, lbB = white_noise_check(retB, "retB")
# acfC, pacfC, lbC = white_noise_check(retC, "retC")



## 2(b) ARMA(1,1) for each return (regardless of 2a findings)


In [ ]:

def fit_arma11(x: int, name: str) -> int:
    res = ARIMA(x, order=(1,0,1), enforce_stationarity=False, enforce_invertibility=False).fit()
    with open(os.path.join(OUTDIR, f"{name}_ARMA_1_1_summary.txt"), "w") as f:
        f.write(str(res.summary()))
    lb = acorr_ljungbox(res.resid.dropna(), lags=20, return_df=True)
    lb.to_csv(os.path.join(OUTDIR, f"{name}_ARMA_1_1_ljungbox.csv"), index=True)
    _ = plot_acf_pacf(res.resid.dropna(), f"{name}_resid_ARMA(1,1)", nlags=20, outdir=OUTDIR)
    return res



# Example (uncomment and replace with your series)
# resA = fit_arma11(retA, "retA")
# resB = fit_arma11(retB, "retB")
# resC = fit_arma11(retC, "retC")
